# 🖐️ 我的第一个模型训练：MNIST 手写数字识别（NPU 版）

欢迎来到深度学习的世界！本教程将带你从零开始，在 **华为昇腾 NPU** 上训练你的第一个神经网络模型。

> 💡 **核心要点**：在昇腾 NPU 上训练模型非常简单，只需 `import torch_npu` 并指定 `device = "npu"`，其余代码和普通 PyTorch 训练完全一样！

## 你将学到什么？

| 知识点 | 说明 |
|--------|------|
| MNIST 数据集 | 深度学习最经典的"Hello World"数据集 |
| CNN 卷积神经网络 | 专门处理图像的神经网络 |
| 模型训练流程 | 数据 → 模型 → 训练 → 评估 → 保存 |
| SwanLab | 训练过程可视化工具 |
| 昇腾 NPU | 华为 AI 加速器 |

## 什么是 MNIST？

MNIST 是一个手写数字图片数据集，包含 60,000 张训练图片和 10,000 张测试图片。
每张图片是 28×28 像素的灰度图，标签是 0~9 的数字。

我们的目标：通过MNIST数据集，训练一个模型，让该模型"看"一张手写数字图片，说出它是哪个数字！

## 📦 第一步：初始化 NPU 环境

In [ ]:
import os, subprocess, warnings
warnings.filterwarnings('ignore')

# ⚠️ 必须在 import torch 之前加载 CANN 环境
# Jupyter 内核不继承 shell 环境变量，缺这步会导致 torch_npu 加载失败
_cann = os.environ.get("ASCEND_TOOLKIT_HOME", "/usr/local/Ascend/ascend-toolkit") + "/set_env.sh"
for line in subprocess.run(["bash", "-c", f"source {_cann} && env"], capture_output=True, text=True).stdout.splitlines():
    if "=" in line:
        k, v = line.split("=", 1)
        os.environ[k] = v

print(f"✅ CANN 环境已加载 (ASCEND_TOOLKIT_HOME={os.environ.get('ASCEND_TOOLKIT_HOME','?')})")

## 📦 第二步：导入工具库

现在 CANN 环境已就绪，导入所有需要的库。

| 库 | 作用 |
|----|------|
| `torch` | PyTorch 深度学习框架 |
| `torch_npu` | 让 PyTorch 能在昇腾 NPU 上运行 |
| `torchvision` | 提供图像数据集和图像处理工具 |
| `swanlab` | 训练过程可视化 |

> 💡 除了 `import torch_npu`，其余导入和普通 PyTorch 项目完全一样。在昇腾 NPU 上训练，就是这么简单。

In [ ]:
import time
import torch
from torch import nn, optim, utils
import torch.nn.functional as F
import torchvision
from torchvision.datasets import MNIST
from torchvision.transforms import ToTensor
import swanlab

import torch_npu  # 让 PyTorch 支持昇腾 NPU

print("✅ 所有库导入成功！")
print(f"   PyTorch 版本: {torch.__version__}")
print(f"   torch_npu 版本: {torch_npu.__version__}")

## 🖥️ 第三步：检查 NPU 设备

在开始训练之前，先确认 NPU 设备可用。

In [ ]:
# 检测 NPU 设备
if torch_npu.npu.is_available():
    device = "npu"
else:
    device = "cpu"

print(f"NPU 是否可用: {torch_npu.npu.is_available()}")
print(f"NPU 设备名称: {torch_npu.npu.get_device_name(0)}")
print(f"NPU 设备数量: {torch_npu.npu.device_count()}")
print(f"使用的设备: {device}")

# 打印张量实际所在的设备，确认真的在 NPU 上
_test_tensor = torch.tensor([1.0]).to(device)
print(f"实际计算设备: {_test_tensor.device}")

## 📚 第四步：加载并可视化 MNIST 数据集

MNIST 数据集包含手写数字 0~9 的图片。让我们加载并看看数据长什么样。

### ToTensor() 是什么？

`ToTensor()` 是 torchvision 提供的图像转换工具，做两件事：

| 操作 | 转换前 | 转换后 |
|------|--------|--------|
| **格式转换** | PIL 图片（H×W×C，0~255 整数） | PyTorch 张量（C×H×W，0~1 浮点数） |
| **归一化** | 像素值范围 0~255 | 像素值范围 0.0~1.0 |

具体来说：
1. **维度顺序调整**：PIL 图片是 `(高, 宽, 通道)`，PyTorch 需要 `(通道, 高, 宽)`
2. **数值类型转换**：从 uint8 整数变为 float32 浮点数
3. **缩放到 0~1**：每个像素除以 255.0，让数值落在 0~1 范围

> 💡 **为什么要归一化到 0~1？** 神经网络对小数值更稳定。如果输入是 0~255 的大整数，
> 梯度会爆炸或消失，模型很难训练。归一化到 0~1 后，训练更稳定、收敛更快。

```python
# 示例：一张 MNIST 图片的转换过程
# 转换前: PIL Image, shape=(28, 28), 像素值 0~255
# 转换后: Tensor,     shape=(1, 28, 28), 像素值 0.0~1.0
transform=ToTensor()
```

In [ ]:
import matplotlib.pyplot as plt

# download=True 会自动检查数据是否已存在，存在则跳过，不存在才下载
dataset = MNIST(
    root=os.getcwd(),
    train=True,
    download=True,
    transform=ToTensor()
)

print(f"训练集大小: {len(dataset)} 张图片")
print(f"图片形状: {dataset[0][0].shape}  (通道数, 高, 宽)")
print(f"标签范围: 0 ~ 9")

# 可视化前 16 张图片
# plt.subplots(行数, 列数, figsize=(宽, 高))
#   2 行 8 列 = 16 个子图，刚好放 16 张图片
#   figsize=(16, 4): 整张图的宽度 16 英寸、高度 4 英寸（宽高比 4:1，适合横排展示）
fig, axes = plt.subplots(2, 8, figsize=(16, 4))
for i, ax in enumerate(axes.flat):
    image, label = dataset[i]
    ax.imshow(image.squeeze(), cmap='gray')
    ax.set_title(f"Label: {label}", fontsize=12)
    ax.axis('off')
plt.suptitle("MNIST Dataset - First 16 Images", fontsize=16)
plt.tight_layout()
plt.show()

## ✂️ 第五步：划分训练集和验证集

把 60,000 张图片分成两部分：

| 用途 | 数量 | 作用 |
|------|------|------|
| **训练集** | 55,000 | 模型从中学习 |
| **验证集** | 5,000 | 检验模型学得怎么样 |

> 💡 **为什么要分？** 就像考试不能拿练习题考学生一样，要用模型没见过的数据来检验。

In [ ]:
batch_size = 256

train_dataset, val_dataset = utils.data.random_split(dataset, [55000, 5000])
train_dataloader = utils.data.DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_dataloader = utils.data.DataLoader(val_dataset, batch_size=8, shuffle=False)

print(f"训练集: {len(train_dataset)} 张 → {len(train_dataloader)} 个批次 (每批 {batch_size} 张)")
print(f"验证集: {len(val_dataset)} 张 → {len(val_dataloader)} 个批次")

## 🧠 第六步：构建卷积神经网络 (CNN)

### 网络结构与 shape 推导

> **公式提示**
> - Conv2d 输出尺寸：`(W - K + 1)`，其中 W=输入宽，K=卷积核大小
> - MaxPool2d 输出尺寸：`W / S`，其中 W=输入宽，S=池化窗口大小
> - 展平（flatten）：把多维拉成一维，`C×H×W → (C×H×W,)`

```
输入图片
shape: (1, 28, 28)          # 1 通道, 高 28, 宽 28
    │
    ▼
Conv2d(in=1, out=10, kernel=5)    # 28 - 5 + 1 = 24
shape: (10, 24, 24)         # 10 通道, 高 24, 宽 24
    │
    ▼
ReLU                              # 激活函数，不改变 shape
shape: (10, 24, 24)
    │
    ▼
MaxPool2d(kernel=2, stride=2)     # 24 / 2 = 12
shape: (10, 12, 12)         # 空间尺寸减半
    │
    ▼
Conv2d(in=10, out=20, kernel=3)   # 12 - 3 + 1 = 10
shape: (20, 10, 10)         # 20 通道, 高 10, 宽 10
    │
    ▼
ReLU                              # 不改变 shape
shape: (20, 10, 10)
    │
    ▼
Flatten                           # 20 × 10 × 10 = 2000
shape: (2000,)               # 展平为一维向量
    │
    ▼
Linear(in=2000, out=500)          # 全连接层，压缩特征
shape: (500,)
    │
    ▼
ReLU                              # 不改变 shape
shape: (500,)
    │
    ▼
Linear(in=500, out=10)            # 输出 10 个类别的得分
shape: (10,)
    │
    ▼
LogSoftmax                        # 转为 log 概率，不改变 shape
shape: (10,)                # [log P(0), log P(1), ..., log P(9)]
```

### 对应代码

| 层 | 代码 | 输入 shape | 输出 shape |
|---|---|---|---|
| Conv1 | `nn.Conv2d(1, 10, 5)` | (1, 28, 28) | (10, 24, 24) |
| MaxPool | `F.max_pool2d(out, 2, 2)` | (10, 24, 24) | (10, 12, 12) |
| Conv2 | `nn.Conv2d(10, 20, 3)` | (10, 12, 12) | (20, 10, 10) |
| Flatten | `out.view(batch, -1)` | (20, 10, 10) | (2000,) |
| FC1 | `nn.Linear(2000, 500)` | (2000,) | (500,) |
| FC2 | `nn.Linear(500, 10)` | (500,) | (10,) |

In [ ]:
class ConvNet(nn.Module):
    """一个简单的卷积神经网络，用于识别手写数字"""

    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 10, 5)    # 28×28 → 24×24
        self.conv2 = nn.Conv2d(10, 20, 3)   # 12×12 → 10×10
        self.fc1 = nn.Linear(20 * 10 * 10, 500)
        self.fc2 = nn.Linear(500, 10)

    def forward(self, x):
        out = F.relu(self.conv1(x))
        out = F.max_pool2d(out, 2, 2)
        out = F.relu(self.conv2(out))
        out = out.view(x.size(0), -1)
        out = F.relu(self.fc1(out))
        out = F.log_softmax(self.fc2(out), dim=1)
        return out

model = ConvNet()
print(model)
print(f"\n模型总参数量: {sum(p.numel() for p in model.parameters()):,} 个")

## 🚀 第七步：把模型放到 NPU 上

就像把游戏安装到显卡上才能用显卡跑一样，我们需要把模型"搬"到 NPU 上才能用 NPU 加速训练。

In [ ]:
# 把模型搬到 NPU 上
# .to(device) 把模型所有参数从 CPU 内存搬到 NPU 显存，之后所有计算都在 NPU 上进行
model = model.to(device)

# 检查模型是否真的在 NPU 上
# model.parameters() 返回模型所有参数的迭代器
# next() 取第一个参数，看它的 .device 属性就知道模型在哪里
first_param = next(model.parameters())
print(f"模型参数所在设备: {first_param.device}")
print(f"是否在 NPU 上: {first_param.is_npu}")

## ⚙️ 第八步：定义损失函数和优化器

| 组件 | 选择 | 说明 |
|------|------|------|
| 损失函数 | CrossEntropyLoss | 分类问题的标准选择 |
| 优化器 | Adam | 最常用、最稳定的优化器 |
| 学习率 | 0.0001 | 控制每次参数更新的步长 |

In [ ]:
learning_rate = 1e-4
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

print("损失函数: CrossEntropyLoss")
print(f"优化器: Adam (lr={learning_rate})")

## 📸 第九步：初始化 SwanLab 实验跟踪

[SwanLab](https://swanlab.cn) 会自动记录训练过程中的损失值和准确率，生成漂亮的可视化曲线。

In [ ]:
# 如果重复执行此 Cell，先结束上一个 run，避免 401 错误
try:
    swanlab.finish()
except Exception:
    pass

# 初始化 SwanLab 实验跟踪
#   project: 项目名（同一个项目的多次实验会归在一起）
#   experiment_name: 本次实验的名字
#   mode: "online" 上传到云端，"offline" 仅本地
#   config: 记录超参数，方便后续对比不同配置的实验结果
run = swanlab.init(
    project="MNIST-example",
    experiment_name="PlainCNN-NPU-Tutorial",
    mode="online",
    config={
        "model": "ConvNet",
        "optim": "Adam",
        "lr": learning_rate,
        "batch_size": batch_size,
        "num_epochs": 10,
        "device": device,
    },
)
print("✅ SwanLab 初始化完成！")

## 🔥 第十步：编写训练和验证函数

训练过程就像学生做练习题：
```
for 每一道题 (batch):
    1. 看题 (前向传播: 模型预测答案)
    2. 对答案 (计算损失: 预测 vs 真实标签)
    3. 找错因 (反向传播: 计算梯度)
    4. 改正错误 (优化器更新参数)
```

In [ ]:
def train(model, device, dataloader, optimizer, criterion, epoch, num_epochs):
    """训练一个 epoch"""
    model.train()  # 设为训练模式（启用 Dropout、BatchNorm 等）
    for iter, (inputs, labels) in enumerate(dataloader):
        # 1. 数据搬到 NPU
        inputs, labels = inputs.to(device), labels.to(device)

        # 2. 前向传播：图片 → 模型 → 预测结果
        outputs = model(inputs)

        # 3. 计算损失：预测结果 vs 真实标签，差距多大
        loss = criterion(outputs, labels)

        # 4. 反向传播：算出每个参数该怎么调
        optimizer.zero_grad()  # 先清空上一步的梯度
        loss.backward()        # 反向传播，计算梯度

        # 5. 更新参数：优化器根据梯度调整模型参数
        optimizer.step()

        # 每 20 个批次打印一次进度
        if iter % 20 == 0:
            print(f'Epoch [{epoch}/{num_epochs}], Iter [{iter+1}/{len(dataloader)}], Loss: {loss.item():.4f}')
            swanlab.log({"train/loss": loss.item()})


def test(model, device, dataloader, epoch):
    """验证集上评估准确率"""
    model.eval()  # 设为评估模式（关闭 Dropout 等）
    correct, total = 0, 0
    with torch.no_grad():  # 验证不需要算梯度，节省显存
        for inputs, labels in dataloader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            # 取概率最大的类别作为预测结果
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    accuracy = correct / total
    swanlab.log({"val/accuracy": accuracy}, step=epoch)
    print(f"  📊 验证 Epoch {epoch}, 准确率: {accuracy:.4f} ({correct}/{total})")
    return accuracy


print("✅ 训练和验证函数定义完成！")

## 🏃 第十一步：在 NPU 上开始训练！

一切就绪！训练 **10 个 epoch**，每 2 个 epoch 验证一次。

⏱️ 在 NPU 上大约需要 3 分钟，请耐心等待……

In [ ]:
num_epochs = 1  # 训练轮数：把整个训练集过 12 遍，你可以修改为你希望的轮数

start_time = time.time()
print(f"🚀 训练开始: {time.strftime('%H:%M:%S')}")
print("=" * 60)

# 每个 epoch 把训练集完整过一遍
for epoch in range(1, num_epochs + 1):
    swanlab.log({"train/epoch": epoch}, step=epoch)
    # 训练：遍历所有批次，前向 + 反向 + 更新参数
    train(model, device, train_dataloader, optimizer, criterion, epoch, num_epochs)
    # 每 2 个 epoch 验证一次，看模型在验证集上表现如何
    if epoch % 2 == 0:
        test(model, device, val_dataloader, epoch)

total_time = time.time() - start_time
print("=" * 60)
print(f"✅ 训练完成！总时间: {total_time:.2f}s ({total_time/60:.2f} 分钟)")
swanlab.log({"train/total_time": total_time})

## 💾 第十二步：保存模型

把训练好的模型参数保存到文件，以后可以直接加载使用。

In [ ]:
os.makedirs("checkpoint", exist_ok=True)
torch.save(model.state_dict(), 'checkpoint/latest_checkpoint.pth')
print(f"✅ 模型已保存: checkpoint/latest_checkpoint.pth ({os.path.getsize('checkpoint/latest_checkpoint.pth')/1024:.1f} KB)")

## 🔎 第十三步：用模型预测

让我们用训练好的模型来预测几张验证集的图片！

🟢 绿色 = 预测正确 | 🔴 红色 = 预测错误

In [ ]:
# 从验证集取 16 张图片，用模型预测并可视化
fig, axes = plt.subplots(2, 8, figsize=(16, 5))
correct, total = 0, 16

model.eval()  # 设为评估模式
with torch.no_grad():  # 预测不需要梯度，节省显存
    for i, ax in enumerate(axes.flat):
        image, true_label = val_dataset[i]

        # image 形状是 (1, 28, 28)，unsqueeze(0) 加一维变成 (1, 1, 28, 28)
        # 即 batch_size=1，这样才能送进模型
        output = model(image.unsqueeze(0).to(device))

        # 取概率最大的类别作为预测结果
        _, predicted = torch.max(output, 1)
        pred_label = predicted.item()

        if pred_label == true_label:
            correct += 1

        # 画图片，标题显示真实值和预测值
        ax.imshow(image.squeeze(), cmap='gray')
        color = 'green' if pred_label == true_label else 'red'  # 对=绿，错=红
        ax.set_title(f"True: {true_label}\nPred: {pred_label}", color=color, fontsize=11)
        ax.axis('off')  # 隐藏坐标轴

plt.suptitle(f"Prediction Results ({correct}/{total} correct, accuracy {correct/total:.1%})", fontsize=14)
plt.tight_layout()
plt.show()

## 📊 第十四步：查看训练曲线与分享

SwanLab 自动记录了整个训练过程，点击链接查看可视化曲线。

实验完成后，可以把结果分享给评委/老师查看。

In [ ]:
# 结束 SwanLab 实验，确保数据全部上传
swanlab.finish()

print("🔗 SwanLab 实验面板（可分享给评委查看）:")
print("   https://swanlab.cn/@mlewis/MNIST-example")

## 🎓 总结与下一步

### 恭喜你完成了第一个深度学习模型训练！

你刚刚在 **华为昇腾 NPU** 上训练了一个能识别手写数字的神经网络，准确率达到约 **98%**！

### 关键概念回顾

| 概念 | 一句话解释 |
|------|-----------|
| **数据集** | 模型的"课本"，从中学习知识 |
| **神经网络** | 模型的"大脑"，由很多神经元组成 |
| **卷积层** | 专门提取图像特征的层 |
| **损失函数** | 衡量模型"考试"成绩的评分标准 |
| **优化器** | 模型的"学习方法"，根据错误改进 |
| **Epoch** | 把所有训练数据看一遍叫一个 epoch |
| **Batch** | 每次喂给模型的一小批数据 |
| **学习率** | 模型每次"改正错误"的幅度 |

### 下一步可以尝试

1. 🔧 **调参**：试试不同的学习率、batch_size、epoch 数
2. 🏭 **改模型**：加更多卷积层、改通道数，看效果变化
3. 📊 **换数据集**：试试 FashionMNIST、CIFAR10
4. 📚 **学理论**：深入了解卷积、反向传播的数学原理

---

> 💬 本教程基于 [SwanLab MNIST 官方案例](https://docs.swanlab.cn/examples/mnist.html) 改编，在昇腾 NPU 上训练。

## 📋 FAQ：常见问题

### Q1: torch 和 torch_npu 是什么关系？

`torch` 是 PyTorch 深度学习框架本体，`torch_npu` 是让 torch 能在华为 NPU 上运行的**适配插件**。

```
┌─────────────────────────────────────────┐
│              你的训练代码                 │
├─────────────────────────────────────────┤
│              torch (PyTorch)             │  ← 通用框架：定义模型、训练流程、自动求导
│  nn.Conv2d, nn.Linear, optim.Adam ...    │
├─────────────────────────────────────────┤
│           torch_npu (适配层)              │  ← NPU 后端：把 torch 的算子映射到 NPU 硬件
│  torch_npu.npu, device="npu" ...         │
├─────────────────────────────────────────┤
│           CANN (华为驱动)                │  ← 底层：NPU 驱动 + 算子库
├─────────────────────────────────────────┤
│           Ascend 910B3 (硬件)            │
└─────────────────────────────────────────┘
```

### Q2: 何时用 torch，何时用 torch_npu？

| 场景 | 用什么 | 示例 |
|------|--------|------|
| 定义模型结构 | `torch` | `nn.Conv2d`, `nn.Linear`, `nn.ReLU` |
| 定义损失函数 | `torch` | `nn.CrossEntropyLoss()` |
| 定义优化器 | `torch` | `optim.Adam(model.parameters())` |
| 数据加载 | `torch` | `DataLoader`, `Dataset` |
| 前向/反向传播 | `torch` | `loss.backward()`, `optimizer.step()` |
| 检测 NPU 是否可用 | `torch_npu` | `torch_npu.npu.is_available()` |
| 获取 NPU 设备名 | `torch_npu` | `torch_npu.npu.get_device_name(0)` |
| 指定设备 | 字符串 | `device = "npu"`，`.to("npu")` |

> 💡 简记：**写训练逻辑用 torch，管 NPU 设备用 torch_npu**。

### Q3: NPU 利用率显示 0% 是怎么回事？

**不代表没在用 NPU。** MNIST CNN 模型太小（~28K 参数，4.6MB 显存），910B3 算力巨大，每个 batch 计算只需 ~10ms，AICore 真正高强度的计算只有微秒级，`npu-smi` 的采样窗口抓不到这么短的脉冲。

验证 NPU 确实在用的方法：
- `tensor.device == npu:0` → 数据在 NPU 上 ✅
- `torch.npu.memory_allocated() > 0` → NPU 分配了显存 ✅

> 想看到 AICore 利用率上升，需要更大的模型（如 ResNet50 + CIFAR10），MNIST 这个量级 NPU 就是在"秒杀"每个 batch。

### Q4: 为什么 Jupyter 第一个 Cell 要 source CANN 环境？

Jupyter 内核启动时**不继承** shell 的环境变量（如 `ASCEND_OPP_PATH`）。如果不在 `import torch` 之前设好这些变量，PyTorch 2.7+ 会尝试自动加载 `torch_npu` 后端并失败，留下无法清除的脏状态，导致后续所有 `import torch_npu` 都报错。

## 📝 课后练习

请根据本教程内容完成以下题目进行自测，在每题下方的代码框中输入选项字母后运行。

**第1题**（单选题）`import torch_npu` 的作用是什么？

- A. 安装 NPU 硬件驱动
- B. 让 PyTorch 能在昇腾 NPU 上运行
- C. 加载 MNIST 数据集
- D. 提供训练可视化功能

In [ ]:
q1 = ''  # 填入你的选项，如 'B'，修改后务必运行本单元格（Shift+Enter）
print(f'第1题答案已记录：{q1}' if q1 else '⚠️ 请填入答案并运行本单元格')

**第2题**（单选题）`ToTensor()` 变换的作用是什么？

- A. 把图片转为 PIL 格式
- B. 把图片转为张量并归一化到 [0, 1] 区间
- C. 把图片转为 numpy 数组
- D. 把图片转为 one-hot 编码

In [ ]:
q2 = ''  # 填入你的选项，如 'B'，修改后务必运行本单元格（Shift+Enter）
print(f'第2题答案已记录：{q2}' if q2 else '⚠️ 请填入答案并运行本单元格')

**第3题**（单选题）`nn.Conv2d(1, 10, 5)` 中三个参数分别表示什么？

- A. 输入通道=1, 输出通道=10, 卷积核大小=5
- B. 批大小=1, 学习率=10, 卷积核大小=5
- C. 输入通道=1, 输出通道=10, 步长=5
- D. 批大小=1, 层数=10, 填充=5

In [ ]:
q3 = ''  # 填入你的选项，如 'B'，修改后务必运行本单元格（Shift+Enter）
print(f'第3题答案已记录：{q3}' if q3 else '⚠️ 请填入答案并运行本单元格')

**第4题**（单选题）输入图片尺寸 28×28，经过 `Conv2d(kernel=5)`（无填充）后尺寸变为？

- A. 28×28
- B. 26×26
- C. 24×24
- D. 23×23

In [ ]:
q4 = ''  # 填入你的选项，如 'B'，修改后务必运行本单元格（Shift+Enter）
print(f'第4题答案已记录：{q4}' if q4 else '⚠️ 请填入答案并运行本单元格')

**第5题**（单选题）`F.max_pool2d(out, 2, 2)` 的作用是什么？

- A. 把图片旋转 2 度
- B. 把特征图空间尺寸减半
- C. 把通道数减半
- D. 增加 2 倍特征图大小

In [ ]:
q5 = ''  # 填入你的选项，如 'B'，修改后务必运行本单元格（Shift+Enter）
print(f'第5题答案已记录：{q5}' if q5 else '⚠️ 请填入答案并运行本单元格')

**第6题**（单选题）训练时 `optimizer.zero_grad()` 的作用是？

- A. 清空模型参数
- B. 清空上一步的梯度，防止梯度累加
- C. 把梯度设为 1
- D. 优化模型结构

In [ ]:
q6 = ''  # 填入你的选项，如 'B'，修改后务必运行本单元格（Shift+Enter）
print(f'第6题答案已记录：{q6}' if q6 else '⚠️ 请填入答案并运行本单元格')

**第7题**（单选题）`model.eval()` 和 `model.train()` 的区别是？

- A. 没有区别
- B. eval 关闭 Dropout/BatchNorm，train 开启
- C. eval 只能在 CPU 上运行
- D. train 速度更快

In [ ]:
q7 = ''  # 填入你的选项，如 'B'，修改后务必运行本单元格（Shift+Enter）
print(f'第7题答案已记录：{q7}' if q7 else '⚠️ 请填入答案并运行本单元格')

**第8题**（单选题）验证时使用 `with torch.no_grad()` 的原因是？

- A. 为了提高预测精度
- B. 不需要计算梯度，节省显存和计算时间
- C. 为了让模型更快收敛
- D. 防止模型参数被修改

In [ ]:
q8 = ''  # 填入你的选项，如 'B'，修改后务必运行本单元格（Shift+Enter）
print(f'第8题答案已记录：{q8}' if q8 else '⚠️ 请填入答案并运行本单元格')

**第9题**（单选题）代码中 `device = "npu"` 的作用是？

- A. 指定模型在 CPU 上训练
- B. 指定模型在昇腾 NPU 上训练
- C. 指定使用 numpy 计算
- D. 指定网络参数量

In [ ]:
q9 = ''  # 填入你的选项，如 'B'，修改后务必运行本单元格（Shift+Enter）
print(f'第9题答案已记录：{q9}' if q9 else '⚠️ 请填入答案并运行本单元格')

**第10题**（单选题）在昇腾 NPU 上训练模型时，`torch_npu` 的角色是？

- A. 替代 PyTorch 的全部功能
- B. 作为 PyTorch 的 NPU 后端适配插件
- C. 提供数据集加载功能
- D. 提供模型保存功能

In [ ]:
q10 = ''  # 填入你的选项，如 'B'，修改后务必运行本单元格（Shift+Enter）
print(f'第10题答案已记录：{q10}' if q10 else '⚠️ 请填入答案并运行本单元格')

<details>
<summary>📌 点击查看客观题参考答案</summary>

| 题号 | 答案 | 要点 |
|------|------|------|
| 第1题 | B | torch_npu 是 PyTorch 的昇腾 NPU 适配插件 |
| 第2题 | B | ToTensor 把图片转为张量并归一化到 [0, 1] |
| 第3题 | A | Conv2d(输入通道, 输出通道, 卷积核大小) |
| 第4题 | C | 28 - 5 + 1 = 24 |
| 第5题 | B | MaxPool2d(2,2) 把空间尺寸减半 |
| 第6题 | B | PyTorch 梯度默认累加，每次迭代前需清零 |
| 第7题 | B | eval 关闭 Dropout/BatchNorm 的随机性 |
| 第8题 | B | 验证只做前向传播，不需要梯度 |
| 第9题 | B | device = "npu" 指定模型在昇腾 NPU 上训练 |
| 第10题 | B | torch_npu 是 PyTorch 的 NPU 后端适配插件 |

</details>

## 🔨 实践题

### 🟢 初级：调参实验

修改以下超参数，重新训练模型，观察训练时间和准确率的变化：

| 参数 | 原值 | 试试改成 |
|------|------|---------|
| `learning_rate` | 1e-4 | 1e-3、1e-5 |
| `batch_size` | 256 | 64、512 |
| `num_epochs` | 12 | 5、20 |

**要求**：
1. 每次只改一个参数，记录训练时间和最终准确率
2. 思考：学习率太大或太小会怎样？batch_size 大小对训练有什么影响？

**提示**：修改 Cell 16 的 `learning_rate` 和 Cell 10 的 `batch_size`，重新运行训练 Cell。

### 🟡 中级：改进网络结构

在现有 `ConvNet` 中**添加一个卷积层 + 池化层**，观察模型参数量和准确率的变化。

**要求**：
1. 在 `conv2` 之后、`fc1` 之前，添加 `conv3 = nn.Conv2d(20, 40, 3)` 和一个 `max_pool2d`
2. 修改 `fc1` 的输入维度（需要重新计算 flatten 后的尺寸）
3. 训练并对比：参数量变化了多少？准确率有没有提升？

**提示**：
- 添加 conv3 后 shape 推导：`(20, 10, 10) → conv3 → (40, 8, 8) → pool → (40, 4, 4)`
- `fc1` 输入维度 = 40 × 4 × 4 = 640

### 🔴 高级：用 FashionMNIST 训练

将数据集从 MNIST 换成 **FashionMNIST**（服装分类），调整模型和训练参数，达到 **85% 以上准确率**。

**要求**：
1. 把 `from torchvision.datasets import MNIST` 改为 `from torchvision.datasets import FashionMNIST`
2. 用 FashionMNIST 替换数据加载处的 MNIST
3. 训练模型，记录准确率
4. 可视化预测结果，观察哪些类别容易混淆

**提示**：
- FashionMNIST 图片尺寸也是 28×28，模型结构不用改
- FashionMNIST 的 10 个类别：T恤、裤子、套衫、连衣裙、外套、凉鞋、衬衫、运动鞋、包、踝靴
- 某些类别外观相似（如衬衫 vs 套衫），模型容易混淆，这是正常现象

**加分项**：尝试不同的网络结构（如加 Dropout 防过拟合），对比哪种结构在 FashionMNIST 上效果更好。